# Fig 2d - cross-species exemplar: QKI (human organoid)

QKI shows a conserved developmental 3'UTR-lengthening trajectory along human
cortical-organoid excitatory neurogenesis - directly comparable to the mouse
ortholog Qk shown in Fig 1e (same panel format: smoothed UTR-site proportions as a
stacked area + the per-cell 3'UTR-length index W along pseudotime). QKI is a top
organoid pseudotime hit in BOTH lineages (EXC shown; INH also lengthens, delta-psi
+0.09). Half-width. Kernel: `downstr`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc, seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import rankdata

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

PANEL = "fig2d_xspecies_exemplar"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, S.mm_to_px(56)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)

# Organoid (Knoblich source); excitatory lineage = clean cross-species match to mouse Qk (Fig 1e)
KB_BASE = Path(f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling")
KB_SMOOTH = KB_BASE / "effect_sizes" / "excitatory" / "smoothed_proportions.h5ad"
KB_RANK = KB_BASE / "composite_ranking" / "composite_ranking_optimized2_excitatory.csv"

In [ ]:
def get_gene_data(adata, gene, pt_col, layer=None):
    m = (adata.var["gene_name"].astype(str) == gene).values
    sub = adata[:, m]
    props = sub.layers[layer] if layer else sub.X
    if hasattr(props, "toarray"):
        props = props.toarray()
    props = np.asarray(props, dtype=np.float64)
    utr_sizes = sub.var["utr_size"].values.astype(float)
    order = np.argsort(utr_sizes)  # proximal(small) -> distal(large)
    return props[:, order], utr_sizes[order]


def utr_colormap(k):
    if k <= 2:
        return ["#E24A33", "#4878CF"][:k]
    return list(sns.color_palette("RdYlBu_r", k))


def bin_along_pt(pt, M, n_bins=40, renorm=False):
    v = ~np.isnan(pt)
    ptv, Mv = pt[v], M[v]
    b = pd.cut(ptv, bins=n_bins, labels=False)
    xs, means, sems = [], [], []
    for k in range(n_bins):
        mask = b == k
        n = int(mask.sum())
        if n == 0:
            continue
        xs.append(ptv[mask].mean())
        if Mv.ndim > 1:
            means.append(Mv[mask].mean(axis=0))
            sems.append(
                Mv[mask].std(axis=0, ddof=1) / np.sqrt(n)
                if n > 1
                else np.zeros(Mv.shape[1])
            )
        else:
            means.append(Mv[mask].mean())
            sems.append(Mv[mask].std(ddof=1) / np.sqrt(n) if n > 1 else 0.0)
    xs = np.array(xs)
    means = np.array(means)
    sems = np.array(sems)
    if renorm and means.ndim > 1:
        rs = means.sum(axis=1, keepdims=True)
        rs[rs == 0] = 1
        means = means / rs
    return xs, means, sems


def _ct_comp_strip(ax, pt, ctv, order, palette, x0, x1, y_lo, y_hi, h=0.04, ng=160):
    """Kernel-smoothed cell-type COMPOSITION strip along pseudotime in the band [y_lo, y_hi]
    (replaces the coarse per-cell-type min-max boxes; stacked celltype fractions, Nadaraya-
    Watson smoothed over pseudotime, bandwidth h)."""
    v = np.isfinite(pt)
    ptv, ctvv = pt[v], np.asarray(ctv)[v]
    order = [c for c in order if (ctvv == c).any()] + [
        c for c in np.unique(ctvv) if c not in order
    ]
    grid = np.linspace(x0, x1, ng)
    comp = np.zeros((len(order), ng))
    for j, gp in enumerate(grid):
        w = np.exp(-0.5 * ((ptv - gp) / h) ** 2)
        sw = w.sum() + 1e-9
        for k, c in enumerate(order):
            comp[k, j] = (w * (ctvv == c)).sum() / sw
    comp = comp / (comp.sum(0, keepdims=True) + 1e-9)
    band = y_hi - y_lo
    cum = np.zeros(ng)
    for k, c in enumerate(order):
        ax.fill_between(
            grid,
            y_lo + cum * band,
            y_lo + (cum + comp[k]) * band,
            color=palette.get(c, "#999999"),
            linewidth=0,
        )
        cum = cum + comp[k]


def plot_pair(
    ax_stack,
    ax_wt,
    adata,
    gene,
    pt_col,
    ct_order,
    palette,
    species,
    dpsi,
    padj,
    rhoW,
    layer=None,
    wt_boxes=False,
):
    props, utr_sizes = get_gene_data(adata, gene, pt_col, layer=layer)
    K = props.shape[1]
    colors = utr_colormap(K)
    pt = adata.obs[pt_col].values.astype(float)

    xs, mean_props, _ = bin_along_pt(pt, props, n_bins=40, renorm=True)
    ax_stack.stackplot(xs, mean_props.T, colors=colors, alpha=0.85)
    ax_stack.set_ylim(-0.14, 1.0)
    ax_stack.set_xlim(xs.min(), xs.max())
    ax_stack.set_ylabel("UTR-site proportion", fontsize=S.FONT_LABEL)
    ax_stack.set_title(
        f"$\it{{{gene}}}$  ({species})",
        fontsize=S.FONT_LABEL,
        pad=3,
        color=S.COL_LONG,
        fontweight="bold",
    )
    ax_stack.text(
        0.03,
        0.93,
        f"K = {K}",
        transform=ax_stack.transAxes,
        fontsize=S.FONT_LEGEND,
        va="top",
    )
    obs = adata.obs[[pt_col, "celltype"]].copy()
    obs = obs[np.isfinite(obs[pt_col])]
    if wt_boxes:
        _ct_comp_strip(
            ax_stack,
            obs[pt_col].values.astype(float),
            obs["celltype"].astype(str).values,
            list(ct_order),
            palette,
            xs.min(),
            xs.max(),
            -0.13,
            -0.02,
        )
    else:
        for ct in ct_order:
            sel = obs["celltype"].astype(str) == ct
            if sel.sum() == 0:
                continue
            cp = obs.loc[sel, pt_col]
            ax_stack.fill_between(
                [float(cp.min()), float(cp.max())],
                -0.13,
                -0.02,
                color=palette.get(ct, "0.7"),
                alpha=0.9,
                linewidth=0,
            )
    ax_stack.axhline(0, color="grey", lw=0.5, alpha=0.3)
    ax_stack.set_xticks([])
    ax_stack.spines["bottom"].set_visible(False)

    ranks = (
        (rankdata(utr_sizes, method="average") - 1) / (K - 1)
        if K > 1
        else np.array([0.5])
    )
    W = props @ ranks
    xw, mW, sW = bin_along_pt(pt, W, n_bins=40)
    ax_wt.fill_between(xw, mW - sW, mW + sW, alpha=0.25, color=S.COL_LONG)
    ax_wt.plot(xw, mW, "-o", color=S.COL_LONG, markersize=2, linewidth=1.2)
    ax_wt.annotate(
        "",
        xy=(xw[-1], mW[-1]),
        xytext=(xw[0], mW[0]),
        arrowprops=dict(arrowstyle="->", color=S.COL_LONG, lw=1, alpha=0.4),
    )
    ax_wt.set_ylabel("W (UTR-length index)", fontsize=S.FONT_LABEL)
    ann = (
        r"$\Delta\psi$ = "
        + f"{dpsi:+.2f}"
        + "\n"
        + r"$\rho_W$ = "
        + f"{rhoW:+.2f}"
        + "\n"
        + f"padj = {padj}"
    )
    if wt_boxes:
        # cell-type box strip below the W(t) trajectory (mirrors the stacked-area subpanel;
        # trajectory compressed slightly to fit, no pseudotime ticks). `obs` (finite-pt cells)
        # is reused from the stacked-area section above.
        lo = float((mW - sW).min())
        hi = float((mW + sW).max())
        span = (hi - lo) or 1.0
        _ct_comp_strip(
            ax_wt,
            obs[pt_col].values.astype(float),
            obs["celltype"].astype(str).values,
            list(ct_order),
            palette,
            xw.min(),
            xw.max(),
            lo - 0.13 * span,
            lo - 0.02 * span,
        )
        ax_wt.set_ylim(lo - 0.14 * span, hi)
        ax_wt.axhline(lo, color="grey", lw=0.4, alpha=0.3)
        ax_wt.set_xticks([])
        ax_wt.tick_params(axis="x", bottom=False, labelbottom=False)
        ax_wt.spines["bottom"].set_visible(False)
        ax_wt.text(
            0.04,
            0.96,
            ann,
            transform=ax_wt.transAxes,
            fontsize=S.FONT_ANNOT - 0.5,
            va="top",
            ha="left",
        )
    else:
        ax_wt.set_xlabel("Pseudotime", fontsize=S.FONT_LABEL)
        ax_wt.text(
            0.96,
            0.06,
            ann,
            transform=ax_wt.transAxes,
            fontsize=S.FONT_ANNOT - 0.5,
            va="bottom",
            ha="right",
        )
        ax_wt.set_xticks([0, 0.5, 1.0])
    ax_wt.tick_params(labelsize=S.FONT_TICK)

In [ ]:
kb = sc.read_h5ad(KB_SMOOTH)
kb_rank = pd.read_csv(KB_RANK)
q_kb = kb_rank.loc[kb_rank.gene_name == "QKI"].iloc[0]


def fmt_padj(p):
    e = int(np.floor(np.log10(p)))
    m = p / 10**e
    return f"{m:.1f}x10$^{{{e}}}$"


print(
    "organoid QKI EXC:",
    f"dpsi={q_kb.delta_psi:+.3f} rhoW={q_kb.rho_W:+.3f} padj={q_kb.padj:.1e}",
)

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=FIGSIZE, gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.55}
)
plot_pair(
    axes[0],
    axes[1],
    kb,
    "QKI",
    "dpt_pseudotime_excitatory",
    S.ORGANOID_EXC_ORDER,
    S.paper_palette_organoid_celltypes,
    "Human organoid EXC",
    float(q_kb.delta_psi),
    fmt_padj(float(q_kb.padj)),
    float(q_kb.rho_W),
    layer=None,
)
fig.subplots_adjust(left=0.12, right=0.97, top=0.86, bottom=0.17)
S.save_and_show(fig, PANEL, figure="Fig2")

## Variant: inhibitory exemplar (JPT1), half-width x quarter-height

An inhibitory-lineage exemplar for an inhibitory-only Fig 2: **JPT1**, a clean monotonic
3'UTR-lengthening gene along INH organoid neurogenesis (K = 5 sites, rho_W = +0.74,
delta-psi = +0.23, padj ~ 2e-77 - the top INH lengthening exemplar by the "low-p, decent
effect, clean transition, few sites, lengthening" criteria). Same panel format as the EXC
QKI exemplar (smoothed UTR-site proportions stacked area + per-cell W along pseudotime).
Alternatives if a different gene is preferred: KCNQ2 (ASD/epilepsy), SFPQ (RBP), RNF130.
Saved as `fig2d_xspecies_exemplar__inh_jpt1`.


In [ ]:
# --- Inhibitory exemplar (JPT1): half-width x quarter-height ---
# Inhibitory-lineage counterpart of the EXC QKI panel, for an inhibitory-only Fig 2. JPT1 is
# the top clean INH lengthening exemplar (K=5, monotonic, rho_W=+0.74, dpsi=+0.23, padj~2e-77).
# Reuses plot_pair / get_gene_data / bin_along_pt / fmt_padj and KB_BASE defined above.
KB_INH_SMOOTH = KB_BASE / "effect_sizes" / "inhibitory" / "smoothed_proportions.h5ad"
KB_INH_RANK = (
    KB_BASE / "composite_ranking" / "composite_ranking_optimized2_inhibitory.csv"
)
GENE_INH = "JPT1"

kb_inh = sc.read_h5ad(KB_INH_SMOOTH)
kb_inh_rank = pd.read_csv(KB_INH_RANK)
g_inh = kb_inh_rank.loc[kb_inh_rank.gene_name == GENE_INH].iloc[0]
print(
    f"organoid {GENE_INH} INH: dpsi={g_inh.delta_psi:+.3f} rhoW={g_inh.rho_W:+.3f} "
    f"K={int(g_inh.n_sites)} padj={g_inh.padj:.1e}"
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
    gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.55},
)
plot_pair(
    axes[0],
    axes[1],
    kb_inh,
    GENE_INH,
    "dpt_pseudotime_inhibitory",
    S.ORGANOID_INH_ORDER,
    S.paper_palette_organoid_celltypes,
    "Human organoid INH",
    float(g_inh.delta_psi),
    fmt_padj(float(g_inh.padj)),
    float(g_inh.rho_W),
    layer=None,
)
fig.subplots_adjust(left=0.12, right=0.97, top=0.86, bottom=0.17)
S.save_and_show(fig, "fig2d_xspecies_exemplar__inh_jpt1", figure="Fig2")

## Variants: inhibitory exemplar alternatives + excitatory QKI (half-width x quarter-height)

Two more inhibitory exemplar choices besides JPT1: **GPM6B** (K=4, cleanest/fewest sites,
neuronal; modest Delta-psi +0.13) and **RNF130** (K=7, largest Delta-psi +0.34 but busier /
switch-like). Plus the excitatory **QKI** exemplar re-sized to half-width x quarter-height to
match. Saved as `fig2d_xspecies_exemplar__inh_gpm6b`, `..._inh_rnf130`, `..._exc_qki_quarter`.


In [ ]:
# --- Inhibitory exemplar alternatives (GPM6B, RNF130) + EXC QKI at quarter-height ---
# INH alternatives besides JPT1 (reuse kb_inh, kb_inh_rank); EXC QKI re-sized to match
# (reuse kb, q_kb). All via plot_pair / fmt_padj defined above.
for _g, _stem in [("GPM6B", "gpm6b"), ("RNF130", "rnf130")]:
    _r = kb_inh_rank.loc[kb_inh_rank.gene_name == _g].iloc[0]
    print(
        f"organoid {_g} INH: dpsi={_r.delta_psi:+.3f} rhoW={_r.rho_W:+.3f} "
        f"K={int(_r.n_sites)} padj={_r.padj:.1e}"
    )
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
        gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.55},
    )
    plot_pair(
        axes[0],
        axes[1],
        kb_inh,
        _g,
        "dpt_pseudotime_inhibitory",
        S.ORGANOID_INH_ORDER,
        S.paper_palette_organoid_celltypes,
        "Human organoid INH",
        float(_r.delta_psi),
        fmt_padj(float(_r.padj)),
        float(_r.rho_W),
        layer=None,
    )
    fig.subplots_adjust(left=0.12, right=0.97, top=0.86, bottom=0.17)
    S.save_and_show(fig, f"fig2d_xspecies_exemplar__inh_{_stem}", figure="Fig2")

# Excitatory QKI, re-sized to half-width x quarter-height (matches the INH exemplar panels)
fig, axes = plt.subplots(
    1,
    2,
    figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
    gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.55},
)
plot_pair(
    axes[0],
    axes[1],
    kb,
    "QKI",
    "dpt_pseudotime_excitatory",
    S.ORGANOID_EXC_ORDER,
    S.paper_palette_organoid_celltypes,
    "Human organoid EXC",
    float(q_kb.delta_psi),
    fmt_padj(float(q_kb.padj)),
    float(q_kb.rho_W),
    layer=None,
)
fig.subplots_adjust(left=0.12, right=0.97, top=0.86, bottom=0.17)
S.save_and_show(fig, "fig2d_xspecies_exemplar__exc_qki_quarter", figure="Fig2")

## Variants: "uniform" (equal-cell) INH exemplars (JPT1, GPM6B, RNF130)

The same three inhibitory exemplars on the equal-CELL (empirical-CDF quantile) pseudotime
axis - pseudotime replaced by its rank/N so cells are uniformly distributed on x (the dense
early regions get stretched, sparse ones compressed), exactly as in
`fig1e_exemplars__hnrnpa2b1_half_uniform` vs the equal-width `..._half`. Saved as
`fig2d_xspecies_exemplar__inh_{jpt1,gpm6b,rnf130}_uniform`.


In [ ]:
# --- "Uniform" (equal-cell quantile) versions of the 3 INH exemplars (JPT1, GPM6B, RNF130) ---
# Same panels as above but on the equal-CELL pseudotime axis: pseudotime -> empirical-CDF rank
# (u = rank/N) so cells are uniform on x (as in fig1e_exemplars__hnrnpa2b1_half_uniform).
# Reuses kb_inh, kb_inh_rank, plot_pair, fmt_padj.
_pt_raw = kb_inh.obs["dpt_pseudotime_inhibitory"].values.astype(float)
_valid = ~np.isnan(_pt_raw)
_u = np.full_like(_pt_raw, np.nan)
_u[_valid] = (rankdata(_pt_raw[_valid], method="average") - 0.5) / _valid.sum()
kb_inh.obs["u_inhibitory"] = _u

for _g, _stem in [("JPT1", "jpt1"), ("GPM6B", "gpm6b"), ("RNF130", "rnf130")]:
    _r = kb_inh_rank.loc[kb_inh_rank.gene_name == _g].iloc[0]
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
        gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.55},
    )
    plot_pair(
        axes[0],
        axes[1],
        kb_inh,
        _g,
        "u_inhibitory",
        S.ORGANOID_INH_ORDER,
        S.paper_palette_organoid_celltypes,
        "Human organoid INH",
        float(_r.delta_psi),
        fmt_padj(float(_r.padj)),
        float(_r.rho_W),
        layer=None,
    )
    axes[1].set_xlabel("Pseudotime (quantile)", fontsize=S.FONT_LABEL)
    fig.subplots_adjust(left=0.12, right=0.97, top=0.86, bottom=0.17)
    S.save_and_show(fig, f"fig2d_xspecies_exemplar__inh_{_stem}_uniform", figure="Fig2")

### Variant: "uniform" (equal-cell) excitatory QKI exemplar

The excitatory QKI exemplar on the equal-CELL (empirical-CDF quantile) pseudotime axis, matching
the inhibitory uniform panels. Saved as `fig2d_xspecies_exemplar__exc_qki_uniform`.


In [ ]:
# --- "Uniform" (equal-cell) version of the excitatory QKI exemplar ---
# EXC counterpart of the INH uniform exemplars. Reuses kb (EXC smoothed), q_kb, plot_pair, fmt_padj.
_pt_raw_e = kb.obs["dpt_pseudotime_excitatory"].values.astype(float)
_ve = ~np.isnan(_pt_raw_e)
_ue = np.full_like(_pt_raw_e, np.nan)
_ue[_ve] = (rankdata(_pt_raw_e[_ve], method="average") - 0.5) / _ve.sum()
kb.obs["u_excitatory"] = _ue

fig, axes = plt.subplots(
    1,
    2,
    figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
    gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.55},
)
plot_pair(
    axes[0],
    axes[1],
    kb,
    "QKI",
    "u_excitatory",
    S.ORGANOID_EXC_ORDER,
    S.paper_palette_organoid_celltypes,
    "Human organoid EXC",
    float(q_kb.delta_psi),
    fmt_padj(float(q_kb.padj)),
    float(q_kb.rho_W),
    layer=None,
)
axes[1].set_xlabel("Pseudotime (quantile)", fontsize=S.FONT_LABEL)
fig.subplots_adjust(left=0.12, right=0.97, top=0.86, bottom=0.17)
S.save_and_show(fig, "fig2d_xspecies_exemplar__exc_qki_uniform", figure="Fig2")

## Variants: uniform exemplars with cell-type boxes under W(t) too

The uniform (equal-cell) exemplars re-drawn with a cell-type box strip below the W(t) subpanel
as well (mirroring the stacked-area subpanel; the W trajectory is compressed slightly to fit).
Only the uniform versions (no linear-pseudotime variant). Saved as
`fig2d_xspecies_exemplar__exc_qki_uniform_wtbox` and `..._inh_{jpt1,gpm6b,rnf130}_uniform_wtbox`.


In [ ]:
# --- Uniform exemplars with cell-type boxes under W(t) too (wt_boxes=True) ---
# EXC QKI + INH jpt1/gpm6b/rnf130, equal-cell axis. Reuses kb, kb_inh, kb_inh_rank, q_kb,
# plot_pair, fmt_padj. (Re)computes the equal-cell (ECDF) pseudotime columns to be self-contained.
for _adx, _ptc, _uc in [
    (kb, "dpt_pseudotime_excitatory", "u_excitatory"),
    (kb_inh, "dpt_pseudotime_inhibitory", "u_inhibitory"),
]:
    _pp = _adx.obs[_ptc].values.astype(float)
    _vv = ~np.isnan(_pp)
    _uu = np.full_like(_pp, np.nan)
    _uu[_vv] = (rankdata(_pp[_vv], method="average") - 0.5) / _vv.sum()
    _adx.obs[_uc] = _uu


def _uniform_wtbox(adata, gene, ucol, order, species, row, save):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
        gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.55},
    )
    plot_pair(
        axes[0],
        axes[1],
        adata,
        gene,
        ucol,
        order,
        S.paper_palette_organoid_celltypes,
        species,
        float(row.delta_psi),
        fmt_padj(float(row.padj)),
        float(row.rho_W),
        layer=None,
        wt_boxes=True,
    )
    fig.subplots_adjust(left=0.12, right=0.97, top=0.86, bottom=0.17)
    S.save_and_show(fig, save, figure="Fig2")


_uniform_wtbox(
    kb,
    "QKI",
    "u_excitatory",
    S.ORGANOID_EXC_ORDER,
    "Human organoid EXC",
    q_kb,
    "fig2d_xspecies_exemplar__exc_qki_uniform_wtbox",
)
for _g, _stem in [("JPT1", "jpt1"), ("GPM6B", "gpm6b"), ("RNF130", "rnf130")]:
    _r = kb_inh_rank.loc[kb_inh_rank.gene_name == _g].iloc[0]
    _uniform_wtbox(
        kb_inh,
        _g,
        "u_inhibitory",
        S.ORGANOID_INH_ORDER,
        "Human organoid INH",
        _r,
        f"fig2d_xspecies_exemplar__inh_{_stem}_uniform_wtbox",
    )

### W-only 2-up: RNF130 (INH) vs QKI (EXC) trajectory comparison


In [ ]:
# --- W-only 2-up: RNF130 (INH) vs QKI (EXC), the W-vs-(cell-proportion-binned) pseudotime
# subpanel ONLY, side by side. Gene name on top of each in its lineage colour; no shortening/
# lengthening wording; the W trajectory + fill also lineage-coloured. Reuses kb / kb_inh /
# kb_inh_rank / q_kb / get_gene_data / bin_along_pt / _ct_comp_strip / fmt_padj. ---
def plot_wt_only(
    ax,
    adata,
    gene,
    gene_label,
    ucol,
    ct_order,
    lin_col,
    dpsi,
    padj,
    rhoW,
    annot=False,
    show_ylabel=True,
):
    props, utr_sizes = get_gene_data(adata, gene, ucol, layer=None)
    K = props.shape[1]
    pt = adata.obs[ucol].values.astype(float)
    ranks = (
        (rankdata(utr_sizes, method="average") - 1) / (K - 1)
        if K > 1
        else np.array([0.5])
    )
    W = props @ ranks
    xw, mW, sW = bin_along_pt(pt, W, n_bins=40)
    ax.fill_between(xw, mW - sW, mW + sW, alpha=0.22, color=lin_col)
    ax.plot(xw, mW, "-o", color=lin_col, markersize=1.6, linewidth=1.0)
    ax.annotate(
        "",
        xy=(xw[-1], mW[-1]),
        xytext=(xw[0], mW[0]),
        arrowprops=dict(arrowstyle="->", color=lin_col, lw=0.9, alpha=0.4),
    )
    obs = adata.obs[[ucol, "celltype"]].copy()
    obs = obs[np.isfinite(obs[ucol])]
    lo = float((mW - sW).min())
    hi = float((mW + sW).max())
    span = (hi - lo) or 1.0
    _ct_comp_strip(
        ax,
        obs[ucol].values.astype(float),
        obs["celltype"].astype(str).values,
        list(ct_order),
        S.paper_palette_organoid_celltypes,
        xw.min(),
        xw.max(),
        lo - 0.13 * span,
        lo - 0.02 * span,
    )
    ax.set_ylim(lo - 0.14 * span, hi + 0.03 * span)
    ax.axhline(lo, color="grey", lw=0.4, alpha=0.3)
    ax.set_xticks([])
    ax.set_title(
        f"$\\it{{{gene_label}}}$",
        fontsize=S.FONT_LABEL,
        color=lin_col,
        fontweight="bold",
        pad=2,
    )
    if show_ylabel:
        ax.set_ylabel("W (UTR length)", fontsize=S.FONT_LABEL - 1, labelpad=1)
    if annot:
        ax.text(
            0.04,
            0.96,
            r"$\rho_W$ = " + f"{rhoW:+.2f}",
            transform=ax.transAxes,
            fontsize=S.FONT_ANNOT - 1,
            va="top",
            ha="left",
        )
    ax.tick_params(labelsize=S.FONT_TICK - 1)
    for sp in ("top", "right", "bottom"):
        ax.spines[sp].set_visible(False)


_rnf = kb_inh_rank.loc[kb_inh_rank.gene_name == "RNF130"].iloc[0]


def _wt_pair(figsize, save, annot, left=0.235):
    fig, axes = plt.subplots(1, 2, figsize=figsize, gridspec_kw={"wspace": 0.6})
    plot_wt_only(
        axes[0],
        kb_inh,
        "RNF130",
        "RNF130",
        "u_inhibitory",
        S.ORGANOID_INH_ORDER,
        S.paper_palette_lineage["inhibitory"],
        float(_rnf.delta_psi),
        fmt_padj(float(_rnf.padj)),
        float(_rnf.rho_W),
        annot=annot,
        show_ylabel=True,
    )
    plot_wt_only(
        axes[1],
        kb,
        "QKI",
        "QKI",
        "u_excitatory",
        S.ORGANOID_EXC_ORDER,
        S.paper_palette_lineage["excitatory"],
        float(q_kb.delta_psi),
        fmt_padj(float(q_kb.padj)),
        float(q_kb.rho_W),
        annot=annot,
        show_ylabel=False,
    )
    fig.subplots_adjust(left=0.235, right=0.98, top=0.85, bottom=0.06, wspace=0.5)
    S.save_and_show(fig, save, figure="Fig2")


_wt_pair(
    (S.QUARTER_W_IN, S.EIGHTH_H_IN),
    "fig2d_xspecies_W_rnf130_vs_qki__quarter_eighth",
    annot=False,
)
_wt_pair(
    (S.HALF_W_IN, S.QUARTER_H_IN),
    "fig2d_xspecies_W_rnf130_vs_qki__half_quarter",
    annot=True,
    left=0.12,
)

### RNF130 vs QKI: LEFT subpanel (stacked proportions over pseudotime), half-width quarter-height


In [ ]:
# --- RNF130 (INH) vs QKI (EXC): the LEFT subpanel of the exemplars (stacked UTR-site
# proportions over pseudotime + cell-type composition strip). half_quarter + a SQUARE variant
# (half width, height reduced via box_aspect so the 0..1 line-plot region is square). ---
def plot_stack_only(
    ax,
    adata,
    gene,
    gene_label,
    ucol,
    ct_order,
    lin_col,
    show_ylabel=True,
    box_aspect=None,
):
    props, utr_sizes = get_gene_data(adata, gene, ucol, layer=None)
    K = props.shape[1]
    colors = utr_colormap(K)
    pt = adata.obs[ucol].values.astype(float)
    xs, mean_props, _ = bin_along_pt(pt, props, n_bins=40, renorm=True)
    ax.stackplot(xs, mean_props.T, colors=colors, alpha=0.85)
    ax.set_xlim(xs.min(), xs.max())
    obs = adata.obs[[ucol, "celltype"]].copy()
    obs = obs[np.isfinite(obs[ucol])]
    _ct_comp_strip(
        ax,
        obs[ucol].values.astype(float),
        obs["celltype"].astype(str).values,
        list(ct_order),
        S.paper_palette_organoid_celltypes,
        xs.min(),
        xs.max(),
        -0.13,
        -0.02,
    )
    ax.set_ylim(-0.14, 1.0)
    ax.axhline(0, color="grey", lw=0.5, alpha=0.3)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    ax.set_title(
        f"$\\it{{{gene_label}}}$",
        fontsize=S.FONT_LABEL,
        color=lin_col,
        fontweight="bold",
        pad=2,
    )
    if show_ylabel:
        ax.set_ylabel("UTR-site proportion", fontsize=S.FONT_LABEL - 0.5)
    ax.text(
        0.03, 0.94, f"K={K}", transform=ax.transAxes, fontsize=S.FONT_LEGEND, va="top"
    )
    ax.tick_params(labelsize=S.FONT_TICK - 1)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    if box_aspect is not None:
        ax.set_box_aspect(
            box_aspect
        )  # box h/w; 1.14 makes the 0..1 region square (ylim spans 1.14)


_rnf2 = kb_inh_rank.loc[kb_inh_rank.gene_name == "RNF130"].iloc[0]


def _stack_pair(figsize, save, left=0.12, box_aspect=None, top=0.85, bottom=0.06):
    fig, axes = plt.subplots(1, 2, figsize=figsize, gridspec_kw={"wspace": 0.45})
    plot_stack_only(
        axes[0],
        kb_inh,
        "RNF130",
        "RNF130",
        "u_inhibitory",
        S.ORGANOID_INH_ORDER,
        S.paper_palette_lineage["inhibitory"],
        show_ylabel=True,
        box_aspect=box_aspect,
    )
    plot_stack_only(
        axes[1],
        kb,
        "QKI",
        "QKI",
        "u_excitatory",
        S.ORGANOID_EXC_ORDER,
        S.paper_palette_lineage["excitatory"],
        show_ylabel=False,
        box_aspect=box_aspect,
    )
    fig.subplots_adjust(left=left, right=0.98, top=top, bottom=bottom, wspace=0.45)
    S.save_and_show(fig, save, figure="Fig2")


_stack_pair(
    (S.HALF_W_IN, S.QUARTER_H_IN), "fig2d_xspecies_props_rnf130_vs_qki__half_quarter"
)
_stack_pair(
    (S.HALF_W_IN, 1.62),
    "fig2d_xspecies_props_rnf130_vs_qki__half_square",
    box_aspect=1.14,
    top=0.90,
    bottom=0.05,
)

### Unassigned panel: JPT1 (INH) UTR-site proportions per biological replicate (equal-cell)

Left-subplot-only view (stacked UTR-site proportions vs equal-cell pseudotime), one subplot per
uninduced (controls-only) inhibitory biological replicate, to show the JPT1 lengthening is
consistent across the 8 independent replicates. Uses RAW (unsmoothed) counts so replicates are
genuinely independent (the graph-smoothed layer blends neighbours across replicates -> circular),
mirroring Fig 1 `supp_8_exemplars_per_mouse__props_Qk_equalcell`. Saved to unassigned_panels as
`exemplars_per_replicate__props_JPT1_equalcell`.


In [ ]:
# --- Unassigned panels: exemplar UTR-site proportions per biological replicate (equal-cell) ---
# Left-subplot only (stacked site proportions vs equal-cell pseudotime), one subplot per uninduced
# biological replicate, using RAW counts so replicates are INDEPENDENT (smoothing across cells,
# not across replicates). Mirrors Fig1 supp_8_exemplars_per_mouse. Replicates with fewer than
# MIN_COVERED gene-covered cells are dropped. Two rendering modes over the equal-cell (ECDF)
# pseudotime axis: disjoint equal-cell bins (default), or a Gaussian kernel smoother in ECDF
# space (smooth_h != None) -- the kernel averages a ~constant number of cells everywhere (density
# is uniform by construction), giving a smoother line while staying uniform-in-cell-density.
UNASSIGNED_DIR = S.OUT_ROOT / "Fig2" / "unassigned_panels"
UNASSIGNED_DIR.mkdir(parents=True, exist_ok=True)
RAW_ORG_H5 = (
    f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/"
    "merged_bw12_above10cts_apa_named_annotated_fix_pt.h5ad"
)
N_BINS_REP, MIN_COVERED = 20, 150


def _ecdf(x):
    u = np.full_like(x, np.nan, float)
    v = ~np.isnan(x)
    u[v] = (rankdata(x[v], method="average") - 0.5) / v.sum()
    return u


def _bin_means(xc, props, valid, n_bins=N_BINS_REP):
    v = valid & ~np.isnan(xc)
    xv, pv = xc[v], props[v]
    if v.sum() < n_bins:
        n_bins = max(3, int(v.sum()) // 3)
    if n_bins < 1:
        return np.array([]), np.zeros((0, props.shape[1]))
    b = pd.cut(xv, bins=n_bins, labels=False)
    cen, mp = [], []
    for k in range(n_bins):
        mm = b == k
        if mm.sum() > 0:
            cen.append(xv[mm].mean())
            mp.append(pv[mm].mean(0))
    mp = np.array(mp)
    cen = np.array(cen)
    rs = mp.sum(1, keepdims=True)
    rs[rs == 0] = 1
    return cen, mp / rs


def _kernel_smooth(xc, props, valid, h=0.12, n_eval=120):
    """Gaussian-kernel-smoothed site proportions vs the equal-cell (ECDF) axis. Bandwidth `h`
    is in ECDF units (fraction of cells); larger = smoother."""
    x_eval = np.linspace(0.02, 0.98, n_eval)
    v = valid & ~np.isnan(xc)
    uu = xc[v]
    pv = props[v]
    out = np.zeros((n_eval, pv.shape[1]))
    for i, x in enumerate(x_eval):
        w = np.exp(-0.5 * ((uu - x) / h) ** 2)
        sw = w.sum()
        if sw > 0:
            out[i] = (w[:, None] * pv).sum(0) / sw
    rs = out.sum(1, keepdims=True)
    rs[rs == 0] = 1
    return x_eval, out / rs


def _comp_curve(xc, ctv, order, valid, x0, x1, h=0.06, ng=120):
    """Kernel-smoothed cell-type composition (stacked fractions) vs the equal-cell axis."""
    v = valid & ~np.isnan(xc)
    uu = xc[v]
    cc = ctv[v]
    order = [c for c in order if (cc == c).any()] + [
        c for c in np.unique(cc) if c not in order
    ]
    grid = np.linspace(x0, x1, ng)
    comp = np.zeros((len(order), ng))
    for j, gp in enumerate(grid):
        w = np.exp(-0.5 * ((uu - gp) / h) ** 2)
        sw = w.sum() + 1e-9
        for k, c in enumerate(order):
            comp[k, j] = (w * (cc == c)).sum() / sw
    comp = comp / (comp.sum(0, keepdims=True) + 1e-9)
    return grid, comp, order


def _per_replicate_props(
    araw,
    gene,
    lineage_label,
    pt_col,
    ct_order,
    direction,
    save_name,
    n_bins=N_BINS_REP,
    min_covered=MIN_COVERED,
    ncol=4,
    smooth_h=None,
    ct_strip=False,
):
    """Per-replicate stacked UTR-site proportions vs within-replicate equal-cell pseudotime,
    uninduced cells only, RAW counts. `smooth_h` (ECDF-space bandwidth) switches from disjoint
    equal-cell bins to a Gaussian kernel smoother. Grid height scales with #replicates.
    """
    o = araw.obs
    ind = o["induced"].astype(str).values
    samp = o["sample_id"].astype(str).values
    ctv = o["celltype"].astype(str).values
    linv = o["lineage"].astype(str).values
    pt = pd.to_numeric(o[pt_col], errors="coerce").values
    mask = (
        (ind == "False")
        & (linv == lineage_label)
        & np.isfinite(pt)
        & np.isin(ctv, ct_order)
    )
    gm = (araw.var["gene_name"].astype(str) == gene).values
    X = araw[:, gm].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    X = np.asarray(X, float)
    us = araw.var["utr_size"].values[gm].astype(float)
    order = np.argsort(us)
    X = X[:, order]
    site_names = [str(s) for s in araw.var["UTR_site"].values[gm][order]]
    K = X.shape[1]
    tot = X.sum(1)
    valid = tot > 0
    props = np.full_like(X, np.nan)
    props[valid] = X[valid] / tot[valid, None]
    colors = (
        ["#E24A33", "#4878CF"][:K] if K <= 2 else list(sns.color_palette("RdYlBu_r", K))
    )
    covered = pd.Series(
        samp[mask & valid]
    ).value_counts()  # gene-covered cells per replicate
    reps = sorted([s for s in covered.index if covered[s] >= min_covered])
    legend_ct = [c for c in ct_order if c in set(ctv[mask & valid])]
    nrow = int(np.ceil(len(reps) / ncol))
    H_MM = 30 * nrow + (
        30 if ct_strip else 21
    )  # scale height with rows (+room for 2 legends)
    fig, axes = plt.subplots(
        nrow,
        ncol,
        figsize=(S.FULL_W_PX / 300, S.mm_to_px(H_MM) / 300),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes.ravel()
    for i, rep in enumerate(reps):
        ax = axes[i]
        inr = mask & (samp == rep)
        xc = np.full_like(pt, np.nan)
        xc[inr] = _ecdf(pt[inr])
        if smooth_h is None:
            cen, mp = _bin_means(xc, props, valid & inr, n_bins=n_bins)
        else:
            cen, mp = _kernel_smooth(xc, props, valid & inr, h=smooth_h)
        if len(cen):
            ax.stackplot(cen, mp.T, colors=colors, alpha=0.85)
            ax.set_xlim(cen.min(), cen.max())
        if ct_strip:
            _x0, _x1 = (cen.min(), cen.max()) if len(cen) else (0.02, 0.98)
            _grid, _comp, _corder = _comp_curve(
                xc, ctv, ct_order, valid & inr, _x0, _x1
            )
            _ylo, _yhi = -0.14, -0.03
            _band = _yhi - _ylo
            _cum = np.zeros(len(_grid))
            for _k, _c in enumerate(_corder):
                ax.fill_between(
                    _grid,
                    _ylo + _cum * _band,
                    _ylo + (_cum + _comp[_k]) * _band,
                    color=S.paper_palette_organoid_celltypes.get(_c, "#999999"),
                    linewidth=0,
                )
                _cum = _cum + _comp[_k]
            ax.axhline(0, color="grey", lw=0.4, alpha=0.3)
            ax.set_ylim(-0.15, 1.0)
            ax.set_xticks([])
            ax.spines["bottom"].set_visible(False)
        else:
            ax.set_ylim(0, 1)
        ax.set_title(f"{rep}  (n={int((inr & valid).sum())})", fontsize=S.FONT_TICK)
        if i % ncol == 0:
            ax.set_ylabel("Proportion", fontsize=S.FONT_LABEL)
        if (not ct_strip) and i >= (nrow - 1) * ncol:
            ax.set_xlabel("Pseudotime (equal-cell)", fontsize=S.FONT_TICK)
    for j in range(len(reps), len(axes)):
        axes[j].axis("off")
    dc = S.COL_SHORT if direction == "Shortening" else S.COL_LONG
    lin_short = lineage_label.split()[0][:3].upper()
    smooth_tag = f", smoothed h={smooth_h}" if smooth_h is not None else ""
    fig.suptitle(
        f"{gene} ({direction}) - {lin_short} UTR-site proportions per replicate "
        f"[equal-cell, n>={min_covered}{smooth_tag}]",
        fontsize=S.FONT_LABEL,
        color=dc,
        fontweight="bold",
        y=1 - 4.5 / H_MM,
    )
    if ct_strip:
        from matplotlib.patches import Patch as _Patch

        _l1 = fig.legend(
            [plt.Rectangle((0, 0), 1, 1, fc=colors[k]) for k in range(K)],
            site_names,
            loc="lower center",
            ncol=K,
            fontsize=S.FONT_LEGEND - 0.5,
            frameon=False,
            bbox_to_anchor=(0.5, 9.5 / H_MM),
            title="UTR sites (stack)",
            title_fontsize=S.FONT_LEGEND - 0.5,
        )
        fig.add_artist(_l1)
        fig.legend(
            [
                _Patch(facecolor=S.paper_palette_organoid_celltypes.get(c, "#999999"))
                for c in legend_ct
            ],
            legend_ct,
            loc="lower center",
            ncol=len(legend_ct),
            fontsize=S.FONT_LEGEND - 0.5,
            frameon=False,
            bbox_to_anchor=(0.5, 1.2 / H_MM),
            title="cell type (strip)",
            title_fontsize=S.FONT_LEGEND - 0.5,
        )
        fig.subplots_adjust(
            left=0.06,
            right=0.99,
            top=1 - 13.0 / H_MM,
            bottom=19.0 / H_MM,
            hspace=0.6,
            wspace=0.12,
        )
    else:
        fig.legend(
            [plt.Rectangle((0, 0), 1, 1, fc=colors[k]) for k in range(K)],
            site_names,
            loc="lower center",
            ncol=K,
            fontsize=S.FONT_LEGEND - 0.5,
            frameon=False,
            bbox_to_anchor=(0.5, 1.2 / H_MM),
        )
        fig.subplots_adjust(
            left=0.06,
            right=0.99,
            top=1 - 13.0 / H_MM,
            bottom=12.0 / H_MM,
            hspace=0.6,
            wspace=0.12,
        )
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


_araw = sc.read_h5ad(RAW_ORG_H5)
_per_replicate_props(
    _araw,
    "JPT1",
    "Inhibitory neurons",
    "dpt_pseudotime_inhibitory",
    S.ORGANOID_INH_ORDER,
    "Lengthening",
    "exemplars_per_replicate__props_JPT1_equalcell",
)
del _araw

### Unassigned panels: per-replicate equal-cell proportions for RNF130 (INH) and QKI (EXC)

Same per-replicate view as the JPT1 panel (raw counts, within-replicate equal-cell pseudotime,
uninduced controls only) for RNF130 (inhibitory) and QKI (excitatory). Saved to unassigned_panels
as `exemplars_per_replicate__props_RNF130_equalcell` and `..._props_QKI_equalcell`.


In [ ]:
# --- RNF130 (INH) and QKI (EXC) per-replicate equal-cell panels (uninduced, n>=150) ---
# Binned versions of RNF130 (INH) + QKI (EXC), plus a kernel-SMOOTHED RNF130 (h=0.12) since
# RNF130 is sparsely covered and the binned line is jagged. Reuses _per_replicate_props etc.
_araw2 = sc.read_h5ad(RAW_ORG_H5)
_per_replicate_props(
    _araw2,
    "RNF130",
    "Inhibitory neurons",
    "dpt_pseudotime_inhibitory",
    S.ORGANOID_INH_ORDER,
    "Lengthening",
    "exemplars_per_replicate__props_RNF130_equalcell",
)
_per_replicate_props(
    _araw2,
    "RNF130",
    "Inhibitory neurons",
    "dpt_pseudotime_inhibitory",
    S.ORGANOID_INH_ORDER,
    "Lengthening",
    "exemplars_per_replicate__props_RNF130_equalcell_smooth",
    smooth_h=0.12,
)
_per_replicate_props(
    _araw2,
    "QKI",
    "Excitatory neurons",
    "dpt_pseudotime_excitatory",
    S.ORGANOID_EXC_ORDER,
    "Lengthening",
    "exemplars_per_replicate__props_QKI_equalcell",
)
_per_replicate_props(
    _araw2,
    "QKI",
    "Excitatory neurons",
    "dpt_pseudotime_excitatory",
    S.ORGANOID_EXC_ORDER,
    "Lengthening",
    "exemplars_per_replicate__props_QKI_equalcell_smooth",
    smooth_h=0.12,
)
# NEW variant: cell-type composition strip instead of the pseudotime x-axis (+ celltype legend)
_per_replicate_props(
    _araw2,
    "QKI",
    "Excitatory neurons",
    "dpt_pseudotime_excitatory",
    S.ORGANOID_EXC_ORDER,
    "Lengthening",
    "exemplars_per_replicate__props_QKI_equalcell_smooth_ctstrip",
    smooth_h=0.12,
    ct_strip=True,
)
del _araw2